# ClarifySQL · Notebook 04 — The agent and the baselines on the validation episodes

**Goal.** Run every system that does not need a trained adapter on all validation episodes (after the team's review),
with the same SQL writer, tools, simulated user and limits:

| system | what it does |
|---|---|
| B1 / B2 | the request goes straight to the SQL writer (B2: plus up to 2 repairs) |
| oracle | the original, fully specified question goes to the SQL writer (upper bound) |
| B3 | the full agent; the policy is the base model with instructions **and worked examples** |
| B3-zero | the same, with instructions only (a prompt variant) |

Validation results are used to choose B3's prompt and to find problems **before** training. Test episodes
are not touched until the final evaluation.

**Time:** about 25 minutes. **GPU quota:** about 25 minutes.
**Settings:** Kaggle → *Session options* → **GPU T4 x2**, **Internet On**. Colab → *T4 GPU*.

In [ ]:
!nvidia-smi

## 1. Get the project code

This cell clones the project from GitHub (branch `main`) into the notebook's working folder and makes
the `clarifysql` package importable. Running it again later just pulls the newest commits.

*Private repository?* Add a secret called `GITHUB_TOKEN` (Kaggle: **Add-ons → Secrets**; Colab: the
**key icon** in the left bar) and set `USE_TOKEN = True`. Our repository is public, so this is not needed.

In [ ]:
import os, sys, subprocess

REPO_URL = "https://github.com/Sathviktr/LMA_ClarifySQL.git"
BRANCH = "main"
USE_TOKEN = False  # only needed for a private repository

ON_KAGGLE = os.path.exists("/kaggle/working")
WORK = "/kaggle/working" if ON_KAGGLE else "/content"
REPO = f"{WORK}/LMA_ClarifySQL"

if USE_TOKEN:
    if ON_KAGGLE:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    else:
        from google.colab import userdata
        token = userdata.get("GITHUB_TOKEN")
    REPO_URL = REPO_URL.replace("https://", f"https://{token}@")

if os.path.exists(REPO):
    subprocess.run(["git", "-C", REPO, "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, REPO], check=True)
print(subprocess.run(["git", "-C", REPO, "log", "--oneline", "-1"], capture_output=True, text=True).stdout)

sys.path.insert(0, f"{REPO}/src")
print("Running on:", "Kaggle" if ON_KAGGLE else "Colab / other", "| repo at", REPO)

In [ ]:
# Libraries that are not preinstalled (or too old) on Kaggle/Colab. PyTorch with CUDA is already there.
!pip install -q -r {REPO}/requirements-gpu.txt
!pip install -q -e {REPO} --no-deps  # lets the scripts import clarifysql

## 2. Download Spider (tools, the SQL writer and the scoring use the real databases)

In [ ]:
os.environ["CLARIFYSQL_DATA"] = "/tmp/clarifysql_data"
!python {REPO}/scripts/download_data.py --only spider

## 3. Check the whole pipeline without the model (seconds)

A scripted policy that follows each episode's reference path, a keyword-matching simulated user and a
gold-SQL writer must score **1.0** on every metric. Anything less is a bug in our code, not in the model.

In [ ]:
!python {REPO}/scripts/run_agent.py --system reference --split validation --backend mock
!rm -f {REPO}/results/agent/*_mock*

## 4. Direct baselines B1 / B2 and the oracle (about 3 minutes)

In [ ]:
!python {REPO}/scripts/run_agent.py --system direct --split validation
!python {REPO}/scripts/run_agent.py --system oracle --split validation

## 5. The prompted agent B3, two prompt variants (about 15 minutes)

The model plays three roles: the policy (adapter off), the simulated user (adapter off, two-stage) and the
SQL writer (adapter off). Episodes run in lockstep, so every round is one batched call per role.

In [ ]:
!python {REPO}/scripts/run_agent.py --system b3 --split validation
!python {REPO}/scripts/run_agent.py --system b3zero --split validation

## 6. Comparison table

In [ ]:
import json, glob
cols = ["execution_accuracy", "decision_macro_f1", "unnecessary_question_rate_clear", "correct_abstention_rate",
        "avg_clarification_turns", "invalid_output_rate", "limit_rate"]
print(f"{'system':<14}" + "".join(f"{c[:18]:>20}" for c in cols))
for path in sorted(glob.glob(f"{REPO}/results/agent/validation_*_summary.json")):
    s = json.load(open(path))
    print(f"{s['name']:<14}" + "".join(f"{str(s[c]):>20}" for c in cols))

## 7. Pack the results

In [ ]:
!cd {REPO} && zip -qr {WORK}/checkpoint4_results.zip results/agent
!ls -la {WORK}/checkpoint4_results.zip

## 8. Send the results back

Download `checkpoint4_results.zip` (Kaggle: **Output** panel; Colab: **folder icon**) and unzip it **inside
the repository folder**, so the files land in `results/agent/`. Then **stop the session**.